In [5]:
import torch
from qwen_tts import Qwen3TTSModel
from qwen_tts.core.models import BasicSpeakerEncoder
import soundfile as sf

In [6]:
# 1. Load the pre-trained base pipeline
tts = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16
)

# 2. Instantiate and load your custom speaker embedding model
custom_encoder = BasicSpeakerEncoder()
custom_encoder.load_state_dict(torch.load("checkpoints/BasicSpeakerEncoder_trial_3/best.pt"))

# 3. Swap out the default ECAPA-TDNN encoder for your custom one!
custom_encoder.to(tts.device).to(tts.model.dtype).eval()
tts.model.speaker_encoder = custom_encoder

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 30897.27it/s]


In [7]:
# 4. Generate as usual; it will now use your custom embeddings
wavs, sr = tts.generate_voice_clone(
    text="This is a test of my custom speaker embeddings.",
    language="English",
    ref_audio="./data/LibriSpeech/dev-clean/84/121123/84-121123-0002.flac",
    ref_text="AT THIS MOMENT THE WHOLE SOUL OF THE OLD MAN SEEMED CENTRED IN HIS EYES WHICH BECAME BLOODSHOT THE VEINS OF THE THROAT SWELLED HIS CHEEKS AND TEMPLES BECAME PURPLE AS THOUGH HE WAS STRUCK WITH EPILEPSY NOTHING WAS WANTING TO COMPLETE THIS BUT THE UTTERANCE OF A CRY"
)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [8]:
# 5. Save the Result
output_filename = "vaders_voice_clone.wav"
sf.write(output_filename, wavs[0], sr)

print(f"✅ Success! Audio saved as {output_filename}")

✅ Success! Audio saved as vaders_voice_clone.wav
